#Data Marts

##Route Performance Data Mart

**Purpose:** To pre-aggregate on-time performance, cancellation, and diversion patterns at the Origin-Destination-Carrier level, providing the Network Planning/Scheduling team with actionable route-level insights directly without needing to join 5 dimensions on the flight-grain fact table.

**Target Users**: Airline Network Planning & Scheduling analysts—who identify routes and carriers with chronic delays or cancellations to make decisions regarding schedule buffer adjustments, route discontinuations, or resource reallocations.

**Analytical Benefits**:

* Improved Query Performance: Because data is pre-aggregated, heavy COUNT, AVG, and SUM logic isn't repeatedly executed on every query.

* Optimal Granularity: The Route + Carrier grain aligns directly with how scheduling decisions are made—focusing on recurring services rather than individual flights.

* Self-Service Friendly: With airport codes, airport names, and carrier names available directly, non-technical stakeholders can easily conduct self-service reporting.

In [0]:
from pyspark.sql import functions as F

fact = spark.table("dwbi.gold.fact_flight_performance")

route_mart_stage = fact.groupBy("origin_airport_key", "dest_airport_key", "carrier_key").agg(
    F.count("*").alias("total_flights"),
    F.sum("is_cancelled").alias("cancelled_flights"),
    F.sum("is_diverted").alias("diverted_flights"),
    F.avg(F.when(F.col("is_delay_applicable"), F.col("dep_delay_min"))).alias("avg_dep_delay_min"),
    F.avg(F.when(F.col("is_delay_applicable"), F.col("arr_delay_min"))).alias("avg_arr_delay_min"),
    F.avg(
        F.when(F.col("is_delay_applicable") & (F.col("arr_delay_min") <= 15), 1)
         .when(F.col("is_delay_applicable"), 0)
    ).alias("on_time_rate")
).withColumn(
    "cancellation_rate", F.col("cancelled_flights") / F.col("total_flights")
)

In [0]:
dim_airport = spark.table("dwbi.gold.dim_airport")
dim_carrier = spark.table("dwbi.gold.dim_carrier")

origin_info = dim_airport.select(
    F.col("airport_key").alias("origin_airport_key"),
    F.col("airport_code").alias("origin_code"),
    F.col("airport_city").alias("origin_city")
)
dest_info = dim_airport.select(
    F.col("airport_key").alias("dest_airport_key"),
    F.col("airport_code").alias("dest_code"),
    F.col("airport_city").alias("dest_city")
)

route_mart = route_mart_stage \
    .join(origin_info, "origin_airport_key", "left") \
    .join(dest_info, "dest_airport_key", "left") \
    .join(dim_carrier, "carrier_key", "left") \
    .select(
        "origin_code", "origin_city", "dest_code", "dest_city",
        "carrier_code", "carrier_name",
        "total_flights", "cancelled_flights", "diverted_flights",
        "cancellation_rate", "avg_dep_delay_min", "avg_arr_delay_min", "on_time_rate"
    )

route_mart.createOrReplaceTempView("route_mart_view")

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS dwbi.mart;

CREATE OR REPLACE TABLE dwbi.mart.route_performance AS
SELECT * FROM route_mart_view;

num_affected_rows,num_inserted_rows


In [0]:
mart = spark.table("dwbi.mart.route_performance")
print("Total routes:", mart.count())
print("Distinct route+carrier combos:", mart.select("origin_code", "dest_code", "carrier_code").distinct().count())

Total routes: 9686
Distinct route+carrier combos: 9686


###Airport Operations Data Mart

**Purpose**: To track the daily operational stress level of an airport by combining daily departure + arrival traffic, cancellations, and delays per airport.

**Target Users**: Airport Ground Operations team—unlike the Network Planning team using the Route Performance mart (which focuses on route-level, long-term scheduling), this team makes daily decisions regarding resource, gate, and crew allocations.

**Analytical Benefits**:

* Operational-Level Granularity: Provides a grain that directly answers "How are operations at this airport today?"—something that cannot be easily derived from a route-level mart.

* Proactive Operational Adjustments: Enables quick identification of days with high cancellations or delays at specific airports, allowing teams to proactively arrange weather-contingency plans or staffing adjustments.

* Clear Delay Attribution: By displaying departures and arrivals separately, teams can easily determine whether delays stem from local departure issues or inbound arrival traffic.

In [0]:
from pyspark.sql import functions as F

fact = spark.table("dwbi.gold.fact_flight_performance")

departure_stats = fact.groupBy("origin_airport_key", "date_key").agg(
    F.count("*").alias("total_departures"),
    F.sum("is_cancelled").alias("cancelled_departures"),
    F.sum("is_diverted").alias("diverted_departures"),
    F.avg(F.when(F.col("is_delay_applicable"), F.col("dep_delay_min"))).alias("avg_dep_delay_min")
).withColumnRenamed("origin_airport_key", "airport_key")

In [0]:
arrival_stats = fact.groupBy("dest_airport_key", "date_key").agg(
    F.count("*").alias("total_arrivals"),
    F.sum("is_cancelled").alias("cancelled_arrivals"),
    F.sum("is_diverted").alias("diverted_arrivals"),
    F.avg(F.when(F.col("is_delay_applicable"), F.col("arr_delay_min"))).alias("avg_arr_delay_min")
).withColumnRenamed("dest_airport_key", "airport_key")

In [0]:
airport_ops_stage = departure_stats.join(
    arrival_stats, on=["airport_key", "date_key"], how="full_outer"
).fillna(0, subset=[
    "total_departures", "cancelled_departures", "diverted_departures",
    "total_arrivals", "cancelled_arrivals", "diverted_arrivals"
]).withColumn(
    "total_operations", F.col("total_departures") + F.col("total_arrivals")
).withColumn(
    "total_cancellations", F.col("cancelled_departures") + F.col("cancelled_arrivals")
).withColumn(
    "cancellation_rate", F.col("total_cancellations") / F.col("total_operations")
)

In [0]:
dim_airport = spark.table("dwbi.gold.dim_airport")
dim_date = spark.table("dwbi.gold.dim_date")

airport_ops_mart = airport_ops_stage \
    .join(dim_airport, "airport_key", "left") \
    .join(dim_date, "date_key", "left") \
    .select(
        "airport_code", "airport_name", "airport_city",
        "full_date", "day_of_week", "is_weekend",
        "total_departures", "total_arrivals", "total_operations",
        "cancelled_departures", "cancelled_arrivals", "total_cancellations", "cancellation_rate",
        "diverted_departures", "diverted_arrivals",
        "avg_dep_delay_min", "avg_arr_delay_min"
    )

airport_ops_mart.createOrReplaceTempView("airport_ops_mart_view")

In [0]:
%sql
CREATE OR REPLACE TABLE dwbi.mart.airport_operations AS
SELECT * FROM airport_ops_mart_view;

num_affected_rows,num_inserted_rows


In [0]:
mart2 = spark.table("dwbi.mart.airport_operations")
print("Total rows:", mart2.count())
print("Distinct airport+date combos:", mart2.select("airport_code", "full_date").distinct().count())

Total rows: 10034
Distinct airport+date combos: 10034
